# Sesión 06 - Lab 1: Manipulación de columnas, filas y estructuras anidadas

Este laboratorio trabaja con un extracto de interacciones de soporte técnico (`interacciones_soporte.json`) del mismo CRM interno de la Sesión 05: cada ticket trae un array anidado de `mensajes` (el hilo completo de la conversación), y algunos tickets llegaron reingestados más de una vez. A diferencia de la Sesión 05, acá el foco no es limpiar nulos sino manipular la estructura del dato: aplanar el array de mensajes con `explode`/`posexplode`, dividir y renombrar columnas, y filtrar con condiciones compuestas. La deduplicación de los tickets repetidos queda para el Lab 2. Este notebook trabaja sobre Bronze tal como llegó, duplicados incluidos, porque no afectan la mecánica de aplanar la estructura.

## Verificación del entorno

In [0]:
# dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_06")
dbutils.fs.ls("abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/sesion06")

## Lab 1-Bronze: Aterrizar las interacciones de soporte en Bronze

El archivo está en formato JSON Lines (un objeto por línea), con un array anidado `mensajes` (un `struct` con `autor`/`texto`/`timestamp` por cada elemento). Igual que en la Sesión 02, se define un `StructType` explícito en vez de dejar que Spark infiera el esquema de una estructura anidada.

In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType, ArrayType
)

schema_mensaje = StructType([
    StructField("autor", StringType(), True),
    StructField("texto", StringType(), True),
    StructField("timestamp", TimestampType(), True),
])

schema_ticket = StructType([
    StructField("ticket_id", StringType(), False),
    StructField("cliente_id", IntegerType(), True),
    StructField("canal", StringType(), True),
    StructField("prioridad", StringType(), True),
    StructField("fecha_apertura", TimestampType(), True),
    StructField("ultima_actualizacion", TimestampType(), True),
    StructField("estado", StringType(), True),
    StructField("categoria_producto", StringType(), True),
    StructField("mensajes", ArrayType(schema_mensaje), True), # arreglo con datos del tipo struct
])

# Controlamos el tipado con los schemas
df_crudo = spark.read.schema(schema_ticket).json(
    # "/Volumes/dbassociate/default/vol_landing/sesion_06/interacciones_soporte.json"
    "abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/sesion06/interacciones_soporte.json"
)

df_bronze_nuevo = (
    df_crudo
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("crm_soporte"))
    .withColumn("batch_id", lit("carga_" + datetime.now().strftime("%Y%m%d_%H%M")))
)

# insertamos en la tabla
df_bronze_nuevo.write.mode("overwrite").saveAsTable("dbassociate.bronze.interacciones_soporte")

print("Tickets aterrizados en Bronze:", df_bronze_nuevo.count())

## Lab 1A: Explorar la estructura anidada sin explode

Un `struct` (como `cliente.direccion` en la Sesión 02) es una casilla con subcasillas fijas; un array (como `mensajes` acá) es una lista de largo variable. Para mirar un elemento puntual del array alcanza con indexarlo (`mensajes[0]`), sin necesidad de aplanar toda la tabla. `size()` cuenta cuántos mensajes tiene cada ticket sin cambiar el grano.

In [0]:
df_bronze = spark.table("dbassociate.bronze.interacciones_soporte")

df_bronze.printSchema()

from pyspark.sql.functions import size

df_bronze
    .select(
        "ticket_id", "canal", "prioridad", size("mensajes").alias("num_mensajes")
    )
    .orderBy(col("num_mensajes").desc())
    .show(10, truncate=False)

df_bronze
    .select(
        "ticket_id",
        col("mensajes")[0]["texto"].alias("primer_mensaje_texto") # tomamos el primer mensaje del arreglo de mensajes
    )
    .show(5, truncate=False)

## Lab 1B: Aplanar el hilo de mensajes con explode()

`explode()` convierte cada mensaje del array en una fila independiente: el grano pasa de "un ticket" a "un mensaje de un ticket". Spark recorre el array en su orden original, pero `explode()` no te entrega ese orden como una columna a la que aferrarte: si el análisis necesita saber cuál fue el primer mensaje del hilo, hace falta otra función (Lab 1C).

In [0]:
from pyspark.sql.functions import explode

df_mensajes_explode = df_bronze
    .select(
        "ticket_id", "cliente_id", "canal", "prioridad", "estado", "categoria_producto",
        explode("mensajes").alias("mensaje") # explode al arreglo con jsons
    ).select(
        "ticket_id", "cliente_id", "canal", "prioridad", "estado", "categoria_producto",
        col("mensaje.autor").alias("autor"),
        col("mensaje.texto").alias("texto"),
        col("mensaje.timestamp").alias("timestamp_mensaje"),
    )


print("Tickets en Bronze:", df_bronze.count())
print("Filas tras el explode (una por mensaje):", df_mensajes_explode.count())

# este ticket tiene 3 mensajes, así que con explode estos se dividieron en 3 filas. Las otras columnas seleccionadas se repeten en las 3 filas.
df_mensajes_explode.filter(col("ticket_id") == "TCK-3007").show(truncate=False)

## Lab 1C: Conservar el orden del hilo con posexplode()

`posexplode()` agrega una columna adicional (`pos`) con la posición del elemento dentro del array, empezando en 0. Eso permite quedarte, por ejemplo, solo con `pos == 0`: el primer mensaje de cada ticket, que en este dataset es siempre el reclamo original del cliente. Útil para analizar el motivo de contacto sin mezclarlo con las respuestas del agente.

In [0]:
from pyspark.sql.functions import posexplode

df_mensajes_posexplode = df_bronze.select(
    "ticket_id", "cliente_id", "canal", "prioridad", "estado", "categoria_producto",
    posexplode("mensajes").alias("pos", "mensaje") # genera una columna adicional con la posición del mensaje
).select(
    "ticket_id", "cliente_id", "canal", "prioridad", "estado", "categoria_producto",
    "pos",
    col("mensaje.autor").alias("autor"),
    col("mensaje.texto").alias("texto"),
    col("mensaje.timestamp").alias("timestamp_mensaje"),
)

df_mensajes_posexplode.filter(col("ticket_id") == "TCK-3007").orderBy(col("pos")).show()

df_primer_mensaje = df_mensajes_posexplode.filter(col("pos") == 0)

print("Filas totales tras posexplode:", df_mensajes_posexplode.count())
print("Tickets con su primer mensaje:", df_primer_mensaje.count())
df_primer_mensaje.select("ticket_id", "autor", "texto").show(10, truncate=False)

## Lab 1D: Manipulación de columnas

`categoria_producto` llega como un texto compuesto (`Facturacion:FAC-01`): un código de categoría y un código de producto pegados con dos puntos. `split()` lo separa en dos columnas de verdad, más útiles para filtrar o agrupar por separado. De paso, se elimina la columna compuesta original, se renombran dos columnas para que el nombre sea más claro (`texto` → `mensaje_texto`, `autor` → `autor_mensaje`), y se agrega una columna calculada (`es_primer_mensaje`) que resume en un booleano lo que antes había que inferir comparando `pos == 0`.

In [0]:
from pyspark.sql.functions import split

df_manipulado = (
    df_mensajes_posexplode
    .withColumn("categoria", split(col("categoria_producto"), ":").getItem(0))
    .withColumn("codigo_producto", split(col("categoria_producto"), ":").getItem(1))
    .drop("categoria_producto")
    .withColumnRenamed("texto", "mensaje_texto")
    .withColumnRenamed("autor", "autor_mensaje")
    .withColumn("es_primer_mensaje", col("pos") == 0) # boolean
)

df_manipulado.select(
    "ticket_id", "categoria", "codigo_producto", "pos", "autor_mensaje", "mensaje_texto", "es_primer_mensaje"
).show(8, truncate=False)

## Lab 1E: Filtros complejos

Combinar varias condiciones con `&` (AND) y `isin()` responde una pregunta de negocio concreta: "¿qué tickets de canal chat o email, prioridad alta o crítica, siguen sin cerrar?". Útil para priorizar seguimiento. `isin()` evita encadenar varios `==` con OR sobre la misma columna.

In [0]:
df_prioritarios = df_manipulado.filter(
    (col("canal").isin(["chat", "email"]))
    & (col("prioridad").isin(["Alta", "Critica"])) # el not isin es con (~col("prioridad").isin(["Alta", "Critica"]))
    & (col("estado") != "Cerrado") # otra forma es usar <>
)

print("Mensajes en tickets prioritarios sin cerrar:", df_prioritarios.count())
df_prioritarios.select("ticket_id", "canal", "prioridad", "estado").distinct().show(truncate=False)

## Lab 1F: Escribir el resultado

Bronze todavía trae los tickets reingresados (`TCK-3005`, `TCK-3011` idénticos; `TCK-3008`, `TCK-3015`, `TCK-3020` con una versión actualizada), así que esta tabla intermedia hereda esos duplicados a nivel de mensaje. El sufijo `_bruto` deja explícito que todavía no es la tabla final: el Lab 2 se encarga de deduplicar antes de calcular cualquier métrica de negocio.

In [0]:
df_manipulado.write.mode("overwrite").saveAsTable("dbassociate.silver.interacciones_soporte_detalle_bruto")

print("Filas escritas (sin deduplicar):", df_manipulado.count())

## Consulta de validación

In [0]:
spark.sql("""
    SELECT categoria, COUNT(*) AS mensajes, COUNT(DISTINCT ticket_id) AS tickets
    FROM dbassociate.silver.interacciones_soporte_detalle_bruto
    GROUP BY categoria
    ORDER BY mensajes DESC
""").show()

## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.interacciones_soporte")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.interacciones_soporte_detalle_bruto")

print("Tablas temporales de este laboratorio eliminadas.")